In [13]:
# %%
import pandas as pd
import numpy as np
from sqlalchemy import create_engine
import urllib.parse
from sklearn.mixture import GaussianMixture

# =========================
# 0) DB 연결
# =========================
DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "dbname": "postgres",
    "user": "postgres",
    "password": "leejangwoo1!",
}

def get_engine(config=DB_CONFIG):
    user = config["user"]
    password = urllib.parse.quote_plus(config["password"])
    host = config["host"]
    port = config["port"]
    dbname = config["dbname"]
    url = f"postgresql+psycopg2://{user}:{password}@{host}:{port}/{dbname}"
    return create_engine(url)

engine = get_engine()

# =========================
# 1) Vision1/2에서 MES 바코드 조회 완료만 로드 + station 생성
# =========================
q_v1 = """
SELECT end_day, contents, time
FROM d1_machine_log."Vision1_machine_log"
WHERE contents = 'MES 바코드 조회 완료'
"""
q_v2 = """
SELECT end_day, contents, time
FROM d1_machine_log."Vision2_machine_log"
WHERE contents = 'MES 바코드 조회 완료'
"""

df_v1 = pd.read_sql(q_v1, engine)
df_v1["station"] = "Vision1"

df_v2 = pd.read_sql(q_v2, engine)
df_v2["station"] = "Vision2"

df = pd.concat([df_v1, df_v2], ignore_index=True)
df = df[["end_day", "station", "contents", "time"]].copy()

# =========================
# 2) shift 경계 제거 (08:20~08:30, 20:20~20:30)
# =========================
df["time_td"] = pd.to_timedelta(df["time"])

t1_start = pd.to_timedelta("08:20:00")
t1_end   = pd.to_timedelta("08:30:00")
t2_start = pd.to_timedelta("20:20:00")
t2_end   = pd.to_timedelta("20:30:00")

mask_shift = (
    ((df["time_td"] >= t1_start) & (df["time_td"] <= t1_end)) |
    ((df["time_td"] >= t2_start) & (df["time_td"] <= t2_end))
)
df = df[~mask_shift].copy()

# 정렬
df = df.sort_values(["end_day", "station", "time_td"]).reset_index(drop=True)

# =========================
# 3) CT 계산 (그룹: end_day+station, diff 절대값, 소수 2자리)
#    - 첫 행 ct는 NaN
#    - ct > 600 제거
# =========================
df["ct"] = df.groupby(["end_day", "station"])["time_td"].diff().dt.total_seconds().abs().round(2)
df = df[(df["ct"].isna()) | (df["ct"] <= 600)].copy()

# 출력용 time 포맷 hh:mm:ss.ss
def format_time(td):
    if pd.isna(td):
        return None
    total = td.total_seconds()
    h = int(total // 3600)
    m = int((total % 3600) // 60)
    s = total % 60
    return f"{h:02d}:{m:02d}:{s:05.2f}"

df["time"] = df["time_td"].apply(format_time)
df = df.drop(columns=["time_td"])

# id 생성 (병합 키)
df.insert(0, "id", range(1, len(df) + 1))

# =========================
# 4) Station별 GMM + Vision_only 조건(max(ct) <= 15)
# =========================
def run_gmm_station(df_station, vision_max_ct=15.0):
    data = df_station[df_station["ct"].notna()].copy()
    if len(data) < 4:
        return pd.DataFrame(columns=["id", "cluster", "ct_pattern", "is_abnormal"])

    X = data["ct"].astype(float).to_numpy().reshape(-1, 1)

    gmm = GaussianMixture(n_components=4, random_state=42)
    gmm.fit(X)
    data["cluster"] = gmm.predict(X)

    stats = (
        data.groupby("cluster")
        .agg(mean_ct=("ct", "mean"),
             max_ct=("ct", "max"),
             count=("ct", "size"))
        .reset_index()
    )

    # Vision_only 후보: max_ct <= 15
    vision_candidates = stats[stats["max_ct"] <= vision_max_ct].copy()
    cluster_vision_only = None
    if len(vision_candidates) > 0:
        cluster_vision_only = vision_candidates.sort_values("mean_ct").iloc[0]["cluster"]

    # MES_delay: mean_ct 최대
    cluster_mes_delay = stats.sort_values("mean_ct", ascending=False).iloc[0]["cluster"]

    # Normal: (Vision_only, MES_delay 제외) count 최대
    excluded = {cluster_mes_delay}
    if cluster_vision_only is not None:
        excluded.add(cluster_vision_only)

    remaining = stats[~stats["cluster"].isin(excluded)].copy()
    if len(remaining) == 0:
        remaining = stats[stats["cluster"] != cluster_mes_delay].copy()

    cluster_normal = remaining.sort_values("count", ascending=False).iloc[0]["cluster"]

    # Other: 나머지
    allc = set(stats["cluster"].tolist())
    used = {cluster_normal, cluster_mes_delay}
    if cluster_vision_only is not None:
        used.add(cluster_vision_only)
    leftovers = list(allc - used)
    cluster_other = leftovers[0] if leftovers else cluster_normal

    def map_group(cid):
        if (cluster_vision_only is not None) and (cid == cluster_vision_only):
            return "Vision_only"
        elif cid == cluster_normal:
            return "Normal_FCT+Vision"
        elif cid == cluster_mes_delay:
            return "MES_delay"
        else:
            return "Other_issue"

    data["ct_pattern"] = data["cluster"].apply(map_group)
    data["is_abnormal"] = data["ct_pattern"].isin(["MES_delay", "Other_issue"])

    return data[["id", "cluster", "ct_pattern", "is_abnormal"]]

res = []
for st in ["Vision1", "Vision2"]:
    res.append(run_gmm_station(df[df["station"] == st], vision_max_ct=15.0))

gmm_res = pd.concat(res, ignore_index=True)

df = df.merge(gmm_res, on="id", how="left")

df["pattern_ko"] = df["ct_pattern"].map({
    "Vision_only":        "Vision만 테스트 (정상, ≤15초)",
    "Normal_FCT+Vision":  "정상 (FCT + Vision)",
    "MES_delay":          "MES 문제로 인한 생산 지연 (비정상)",
    "Other_issue":        "기타 문제 (비정상)",
})
df["is_abnormal_ko"] = df["is_abnormal"].map({True:"비정상", False:"정상", None:None})

# =========================
# 5) Station별 군집 요약(mean_df 형태)
# =========================
mean_df_station = (
    df[df["ct"].notna()]
    .groupby(["station", "ct_pattern", "pattern_ko"], as_index=False)
    .agg(
        mean_ct=("ct", "mean"),
        count=("ct", "size"),
        min_ct=("ct", "min"),
        max_ct=("ct", "max"),
    )
)

mean_df_station["mean_ct"] = mean_df_station["mean_ct"].round(2)
mean_df_station["min_ct"]  = mean_df_station["min_ct"].round(2)
mean_df_station["max_ct"]  = mean_df_station["max_ct"].round(2)

mean_df_station = mean_df_station.sort_values(["station", "mean_ct"]).reset_index(drop=True)

print("[DONE] df / mean_df_station 생성 완료")
mean_df_station

[DONE] df / mean_df_station 생성 완료


,station,ct_pattern,pattern_ko,mean_ct,count,min_ct,max_ct
0,Vision1,Normal_FCT+Vision,정상 (FCT + Vision),18.33,84542,6.00,27.62
1,Vision1,Other_issue,기타 문제 (비정상),40.11,18119,27.63,140.23
2,Vision1,MES_delay,MES 문제로 인한 생산 지연 (비정상),293.59,334,140.40,594.23
3,Vision2,Vision_only,"Vision만 테스트 (정상, ≤15초)",13.55,31080,12.76,14.28
4,Vision2,Normal_FCT+Vision,정상 (FCT + Vision),20.64,76088,5.87,30.23
5,Vision2,Other_issue,기타 문제 (비정상),37.67,6850,30.24,62.29
6,Vision2,MES_delay,MES 문제로 인한 생산 지연 (비정상),156.42,889,62.32,598.89
